In [ ]:
from pathlib import Path
import sys

# Agrega la carpeta 'src' al path de búsqueda de Python de forma relativa
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from inf8239_u01.data import download_csv

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

# Descargamos guardando en data/raw/dataset.csv
dest = Path("../data/raw/dataset.csv")
path = download_csv(URL, destination=dest)
print("Dataset descargado y guardado exitosamente en:", path)

Dataset descargado y guardado exitosamente en: ..\data\raw\dataset.csv


In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/dataset.csv")

print("Dimensiones del dataset (filas, columnas):", df.shape)
print("\nTipos de datos detectados:")
print(df.dtypes)
print("\nPrimeras 5 filas:")
display(df.head())
print("\nÚltimas 5 filas:")
display(df.tail())

assert not df.empty, "El DataFrame no debe estar vacío"

Dimensiones del dataset (filas, columnas): (1000, 21)

Tipos de datos detectados:
checking_status             str
duration                  int64
credit_history              str
purpose                     str
credit_amount             int64
savings_status              str
employment                  str
installment_commitment    int64
personal_status             str
other_parties               str
residence_since           int64
property_magnitude          str
age                       int64
other_payment_plans         str
housing                     str
existing_credits          int64
job                         str
num_dependents            int64
own_telephone               str
foreign_worker              str
class                     int64
dtype: object

Primeras 5 filas:


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2



Últimas 5 filas:


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
995,A14,12,A32,A42,1736,A61,A74,3,A92,A101,...,A121,31,A143,A152,1,A172,1,A191,A201,1
996,A11,30,A32,A41,3857,A61,A73,4,A91,A101,...,A122,40,A143,A152,1,A174,1,A192,A201,1
997,A14,12,A32,A43,804,A61,A75,4,A93,A101,...,A123,38,A143,A152,1,A173,1,A191,A201,1
998,A11,45,A32,A43,1845,A61,A73,4,A93,A101,...,A124,23,A143,A153,1,A173,1,A192,A201,2
999,A12,45,A34,A41,4576,A62,A71,3,A93,A101,...,A123,27,A143,A152,1,A173,1,A191,A201,1


In [ ]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
checking_status,str,0,0.0,4
duration,int64,0,0.0,33
credit_history,str,0,0.0,5
purpose,str,0,0.0,10
credit_amount,int64,0,0.0,921
savings_status,str,0,0.0,5
employment,str,0,0.0,5
installment_commitment,int64,0,0.0,4
personal_status,str,0,0.0,4
other_parties,str,0,0.0,3


In [ ]:
TARGET = "class"
DROP_COLUMNS = []  # No hay identificadores artificiales ni columnas de fuga en este dataset

assert TARGET in df.columns, "El target debe existir en el DataFrame"

X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print("Distribución de clases del target:")
print(y.value_counts(dropna=False))

assert y.notna().all(), "El target no debe tener valores ausentes"
assert y.nunique() >= 2, "El target debe tener al menos dos clases"

Distribución de clases del target:
class
1    700
2    300
Name: count, dtype: int64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Identificar automáticamente columnas numéricas y categóricas
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 2. Sub-pipeline para variables numéricas (imputación de mediana + estandarización)
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# 3. Sub-pipeline para variables categóricas (imputación de moda + codificación One-Hot)
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Ensamblador ColumnTransformer global
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print(f"Columnas numéricas detectadas ({len(num_cols)}): {num_cols}")
print(f"Columnas categóricas detectadas ({len(cat_cols)}): {cat_cols}")

Columnas numéricas detectadas (7): ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Columnas categóricas detectadas (13): ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

# 1. Partición estratificada (80% entrenamiento / 20% prueba)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Pipeline de la Línea Base (Dummy)
dummy = Pipeline([
    ("prep", preprocess),
    ("model", DummyClassifier(strategy="most_frequent"))
])

# 3. Pipeline de la SVM con preprocesamiento integrado sin fuga
svm = Pipeline([
    ("prep", preprocess),
    ("model", SVC(C=1.0, gamma="scale", probability=True, random_state=42))
])

# 4. Ajuste y evaluación comparativa
models_dict = {"dummy": dummy, "svm": svm}

for name, model in models_dict.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    score = f1_score(yte, pred, average="macro")
    print(f"F1-macro ({name}):", round(score, 4))

print("\n--- Reporte detallado de Clasificación (SVM) ---")
print(classification_report(yte, svm.predict(Xte), digits=4))

F1-macro (dummy): 0.4118


c:\Users\Ricky\Universidad Autonoma de Santo Domingo\CI-MACDIA - Documents\Maestria IA y Ciencias de DAtos\3er. Modulo\INF8239_U01\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


F1-macro (svm): 0.72

--- Reporte detallado de Clasificación (SVM) ---
              precision    recall  f1-score   support

           1     0.8063    0.9214    0.8600       140
           2     0.7250    0.4833    0.5800        60

    accuracy                         0.7900       200
   macro avg     0.7656    0.7024    0.7200       200
weighted avg     0.7819    0.7900    0.7760       200



### Interpretación de Resultados del Modelo SVM Base (Paso 9)

- **Desempeño global:** El modelo alcanza una exactitud (*accuracy*) de **0.7900** y un promedio armónico balanceado (*F1-macro*) de **0.7200**, superando con solvencia el comportamiento de la línea base (*DummyClassifier*).
- **Desglose de la clase mayoritaria (Clase 1 - Buen riesgo / Pagador):** 
  - Muestra una alta precisión (**0.8063**) y un excelente *recall* (**0.9214**), lo que indica que el clasificador aprende a identificar con facilidad a los clientes financieramente solventes.
- **Desglose de la clase minoritaria (Clase 2 - Mal riesgo / Incumplidor):** 
  - Presenta un *recall* bajo (**0.4833**). Esto significa que de cada 100 solicitantes reales con riesgo de impago, el modelo solo detecta a unos 48, omitiendo a más de la mitad.
- **Implicación operativa (Costo del error):** 
  - Los falsos negativos (clasificar a un mal cliente como buen pagador) representan el error más costoso para la entidad financiera, ya que conllevan el desembolso de créditos incobrables y pérdidas patrimoniales directas. 
  - Para mitigar este sesgo hacia la clase mayoritaria y mejorar la detección de riesgos, el siguiente paso metodológico consistirá en realizar una **búsqueda optimizada de hiperparámetros ($C$ y $\gamma$) mediante validación cruzada (`GridSearchCV`)**.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# 1. Definir la estrategia de validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Configurar la búsqueda de cuadrícula (Grid Search) asegurando el prefijo 'model__' 
# porque el estimador está encapsulado dentro del Pipeline
search = GridSearchCV(
    estimator=svm,  # Reutilizamos el pipeline 'svm' definido en el paso anterior
    param_grid={
        "model__C": [0.1, 1.0, 10.0, 100.0],
        "model__gamma": ["scale", 0.01, 0.1, 1.0]
    },
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

# 3. Ajustar exclusivamente con el conjunto de entrenamiento (Xtr, ytr)
search.fit(Xtr, ytr)

print("Mejores hiperparámetros encontrados:", search.best_params_)
print("Mejor F1-macro promedio en Validación Cruzada:", round(search.best_score_, 4))

# 4. Evaluar el mejor modelo optimizado sobre el conjunto de prueba (Xte, yte)
best_model = search.best_estimator_
test_pred = best_model.predict(Xte)

print("\n--- Reporte de Clasificación (Modelo Optimizado) ---")
print(classification_report(yte, test_pred, digits=4))
print("F1-macro final en Prueba:", round(f1_score(yte, test_pred, average="macro"), 4))